## Importação das Bibliotecas

In [88]:
import pandas as pd
import requests 
import sqlalchemy
import psycopg2
import json
import numpy as np

In [89]:
pd.set_option('display.max_columns', None)

In [90]:
pd.set_option('display.max_colwidth', None)

## Extração do Staging do Verdana

In [91]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Conectar ao banco PostgreSQL usando SQLAlchemy
url = URL.create(
    drivername="postgresql+psycopg2",
    username="admin_pg",
    password="1nbr@ndsIB",
    host="192.168.50.102",
    port=5432,
    database="db_bi_ti"
)
# Criar a engine de conexão
engine = create_engine(url)
# Ler a tabela
df = pd.read_sql('SELECT * FROM f_chamados_verdanadesk', engine)

## Campo de Tratamento dos dados

In [167]:
df.columns = df.columns.str.strip()
df.columns

Index(['id', 'email_requerente', 'titulo', 'status', 'tipo', 'origem_abertura',
       'prioridade', 'categoria', 'categoria_completa', 'localizacao',
       'nome_tecnico', 'grupo', 'tempo_chamado', 'data_criacao',
       'ultima_atualizacao', 'tempo_atribuicao', 'tempo_solucao',
       'data_solucao', 'data_fechamento', 'expirado', 'tempo_resposta',
       'dominio_email', 'codigo_unidade', 'departamento', 'grupo_descricao',
       'subcategoria', 'status_macro', 'tempo_primeira_resposta_horas',
       'tempo_resolucao_horas', 'status_sla', 'sla_previsto_horas',
       'flag_sla_dentro_prazo', 'faixa_sla', 'flag_sem_tecnico'],
      dtype='str')

In [171]:
df['data_abertura_merge'] = (
    df['data_criacao']
    .dt.date
)

df['data_fechamento_merge'] = (
    df['data_fechamento']
    .dt.date
)

In [133]:
campos_texto = [
    'status',
    'grupo',
    'prioridade',
    'nome_tecnico'
]

for col in campos_texto:

    df[col] = (
        df[col]
        .astype(str)
        .str.upper()
        .str.strip()
    )

In [134]:

valores_nao_atribuidos = [
    'NÃO ATRIBUÍDO',
    'NAO ATRIBUIDO',
    'NULL',
    'NONE',
    ''
]

for col in campos_texto:

    df[col] = np.where(
        df[col].isin(valores_nao_atribuidos),
        np.nan,
        df[col]
    )

In [135]:
campos_data = [
    'data_criacao',
    'tempo_atribuicao',
    'tempo_resposta',
    'data_solucao',
    'data_fechamento'
]

In [136]:
for col in campos_data:

    df[col] = pd.to_datetime(
        df[col],
        errors='coerce'
    )

In [137]:
### Tratamento dos Dados de Grupo, fazendo a separação:

# Coluna Area_responsavel
map_area = {
    'TI | SUPORTE LOCAL': 'TI',
    'TI | SUPORTE A LOJAS': 'TI',
    'TI | PROJETOS': 'TI',
    'ADMINISTRAÇÃO DE PESSOAL': 'RH'
}

#Coluna Torre_responsavel
map_torre = {
    'TI | SUPORTE LOCAL': 'INFRAESTRUTURA',
    'TI | SUPORTE A LOJAS': 'SUPORTE LOJAS',
    'TI | SISTEMAS': 'SISTEMAS',
    'TI | PROJETOS': 'PROJETOS',
    'ADMINISTRAÇÃO DE PESSOAL': 'RH OPERAÇÕES'
}

In [138]:
# Criando os campos email_requerente e dominio_email
df = df.rename(columns={'requerente': 'email_requerente'})
df['dominio_email'] = df['email_requerente'].str.split('@').str[1]
df['dominio_email'] = df['dominio_email'].fillna('Não atribuído')
df

,id,email_requerente,titulo,status,tipo,origem_abertura,prioridade,categoria,categoria_completa,localizacao,nome_tecnico,grupo,tempo_chamado,data_criacao,ultima_atualizacao,tempo_atribuicao,tempo_solucao,data_solucao,data_fechamento,expirado,tempo_resposta,dominio_email,codigo_unidade,departamento,grupo_descricao,subcategoria,status_macro,tempo_primeira_resposta_horas,tempo_resolucao_horas,status_sla,sla_previsto_horas,flag_sla_dentro_prazo,faixa_sla,flag_sem_tecnico
0,27636,aira.coelho@inbrands.com.br,Reparos De Pintura Escritório,FECHADO,Requisição,Formcreator,MÉDIA,Pintura - Hub,Pintura - Hub > Reparos Escritório,HUB-SP - CENESP,NaN,MANUTENÇÃO | HUB,Não atribuído,2026-05-18 10:59:13,2026-05-18 10:59:28,NaT,2026-05-25 10:59:13,2026-05-18 10:59:25,2026-05-18 10:59:28,No prazo,NaT,inbrands.com.br,,MANUTENÇÃO,HUB,Reparos Escritório,Fechado,NaN,0.003333,FINALIZADO,8.0,1.0,ATÉ 1H,0
1,27635,keylla.jesus@inbrands.com.br,Cadastro Liberado No Linx E No Wms Segue Bloqueado,NOVO,Incidente,Formcreator,ALTA,Linx,Linx > Erro Em Tela No Linx,CD EMBU,NaN,TI | SISTEMAS,Não atribuído,2026-05-18 10:48:21,2026-05-18 10:48:21,2026-05-18 11:18:21,2026-05-18 17:48:21,NaT,NaT,No prazo,2026-05-18 00:30:00,inbrands.com.br,,TI,SISTEMAS,Erro Em Tela No Linx,Aberto,-10.305833,NaN,NÃO CLASSIFICADO,4.0,NaN,EM ANDAMENTO,0
2,27634,tommy.fortaleza@inbrands.com.br,A Pagina Do General/ Lançamentos Não Esta Abrindo:,NOVO,Incidente,Formcreator,BAIXA,Relatórios,Relatórios > Relatório Não Enviado,TH - OUTLET FORTALEZA,NaN,TI | SISTEMAS,Não atribuído,2026-05-18 10:47:46,2026-05-18 10:47:46,2026-05-18 13:47:46,2026-05-20 08:47:46,NaT,NaT,No prazo,2026-05-18 03:00:00,inbrands.com.br,,TI,SISTEMAS,Relatório Não Enviado,Aberto,-7.796111,NaN,NÃO CLASSIFICADO,24.0,NaN,EM ANDAMENTO,0
3,27633,vr.portoalegre@inbrands.com.br,Lâmpada Queimada,SOLUCIONADO,Incidente,Formcreator,ALTA,Iluminação - Lojas,Iluminação - Lojas > Totalmente Apagada,VR - PORTO ALEGRE,NaN,MANUTENÇÃO | LOJAS,Não atribuído,2026-05-18 10:45:45,2026-05-18 10:51:00,NaT,2026-05-20 17:45:45,2026-05-18 10:51:00,NaT,No prazo,NaT,inbrands.com.br,,MANUTENÇÃO,LOJAS,Totalmente Apagada,Fechado,NaN,0.087500,FINALIZADO,4.0,1.0,ATÉ 1H,0
4,27632,joao.lacerda@inbrands.com.br,Linx Travado,SOLUCIONADO,Incidente,Formcreator,MUITO ALTA,Linx,Linx > Lentidão Linx,HUB-SP - CENESP,FELIPE BARBOSA GONÇALVES,TI | SISTEMAS,Não atribuído,2026-05-18 10:36:54,2026-05-18 10:48:55,2026-05-18 10:51:54,2026-05-18 13:36:54,2026-05-18 10:48:55,NaT,No prazo,2026-05-18 00:15:00,inbrands.com.br,,TI,SISTEMAS,Lentidão Linx,Fechado,-10.365000,0.200278,FINALIZADO,NaN,0.0,ATÉ 1H,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27072,71,salinas.higienopolis@inbrands.com.br,Notas Fiscais Não Aparecem Para Entrada.,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 18:21:29,2025-04-10 12:04:27,2025-04-08 20:21:29,2025-04-09 21:21:29,2025-04-08 18:59:35,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9703.641944,0.635000,FINALIZADO,24.0,1.0,ATÉ 1H,0
27073,70,salinas.higienopolis@inbrands.com.br,Nota Fiscal De Doação,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 17:55:21,2025-04-10 12:04:27,2025-04-08 19:55:21,2025-04-09 20:55:21,2025-04-08 18:58:27,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9704.077500,1.051667,FINALIZADO,24.0,1.0,1H A 4H,0
27074,69,salinas.iguatemi@inbrands.com.br,Peça Não Cadastrada,FECHADO,Requisição,Formcreator,MUITO BAIXA,Não Atribuído,Não Atribuído,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 15:37:26,2025-07-16 16:

In [139]:
inicio = df['data_criacao'].min()
fim = pd.Timestamp.today()

In [140]:
calendario = pd.date_range(
    start=inicio,
    end=fim,
    freq='D'
)

In [141]:
map_dias = {
    'Monday': 'SEGUNDA',
    'Tuesday': 'TERÇA',
    'Wednesday': 'QUARTA',
    'Thursday': 'QUINTA',
    'Friday': 'SEXTA',
    'Saturday': 'SÁBADO',
    'Sunday': 'DOMINGO'
}

In [142]:
map_filial = {
    'MORUMBI SHOPPING': 'UNI001',
    'IBIRAPUERA': 'UNI002',
    'CENTER NORTE': 'UNI003'
}

df['localizacao'] = (
    df['localizacao']
    .astype(str)
    .str.upper()
    .str.strip()
)

df['codigo_unidade'] = (
    df['localizacao']
    .map(map_filial)
)

df

,id,email_requerente,titulo,status,tipo,origem_abertura,prioridade,categoria,categoria_completa,localizacao,nome_tecnico,grupo,tempo_chamado,data_criacao,ultima_atualizacao,tempo_atribuicao,tempo_solucao,data_solucao,data_fechamento,expirado,tempo_resposta,dominio_email,codigo_unidade,departamento,grupo_descricao,subcategoria,status_macro,tempo_primeira_resposta_horas,tempo_resolucao_horas,status_sla,sla_previsto_horas,flag_sla_dentro_prazo,faixa_sla,flag_sem_tecnico
0,27636,aira.coelho@inbrands.com.br,Reparos De Pintura Escritório,FECHADO,Requisição,Formcreator,MÉDIA,Pintura - Hub,Pintura - Hub > Reparos Escritório,HUB-SP - CENESP,NaN,MANUTENÇÃO | HUB,Não atribuído,2026-05-18 10:59:13,2026-05-18 10:59:28,NaT,2026-05-25 10:59:13,2026-05-18 10:59:25,2026-05-18 10:59:28,No prazo,NaT,inbrands.com.br,NaN,MANUTENÇÃO,HUB,Reparos Escritório,Fechado,NaN,0.003333,FINALIZADO,8.0,1.0,ATÉ 1H,0
1,27635,keylla.jesus@inbrands.com.br,Cadastro Liberado No Linx E No Wms Segue Bloqueado,NOVO,Incidente,Formcreator,ALTA,Linx,Linx > Erro Em Tela No Linx,CD EMBU,NaN,TI | SISTEMAS,Não atribuído,2026-05-18 10:48:21,2026-05-18 10:48:21,2026-05-18 11:18:21,2026-05-18 17:48:21,NaT,NaT,No prazo,2026-05-18 00:30:00,inbrands.com.br,NaN,TI,SISTEMAS,Erro Em Tela No Linx,Aberto,-10.305833,NaN,NÃO CLASSIFICADO,4.0,NaN,EM ANDAMENTO,0
2,27634,tommy.fortaleza@inbrands.com.br,A Pagina Do General/ Lançamentos Não Esta Abrindo:,NOVO,Incidente,Formcreator,BAIXA,Relatórios,Relatórios > Relatório Não Enviado,TH - OUTLET FORTALEZA,NaN,TI | SISTEMAS,Não atribuído,2026-05-18 10:47:46,2026-05-18 10:47:46,2026-05-18 13:47:46,2026-05-20 08:47:46,NaT,NaT,No prazo,2026-05-18 03:00:00,inbrands.com.br,NaN,TI,SISTEMAS,Relatório Não Enviado,Aberto,-7.796111,NaN,NÃO CLASSIFICADO,24.0,NaN,EM ANDAMENTO,0
3,27633,vr.portoalegre@inbrands.com.br,Lâmpada Queimada,SOLUCIONADO,Incidente,Formcreator,ALTA,Iluminação - Lojas,Iluminação - Lojas > Totalmente Apagada,VR - PORTO ALEGRE,NaN,MANUTENÇÃO | LOJAS,Não atribuído,2026-05-18 10:45:45,2026-05-18 10:51:00,NaT,2026-05-20 17:45:45,2026-05-18 10:51:00,NaT,No prazo,NaT,inbrands.com.br,NaN,MANUTENÇÃO,LOJAS,Totalmente Apagada,Fechado,NaN,0.087500,FINALIZADO,4.0,1.0,ATÉ 1H,0
4,27632,joao.lacerda@inbrands.com.br,Linx Travado,SOLUCIONADO,Incidente,Formcreator,MUITO ALTA,Linx,Linx > Lentidão Linx,HUB-SP - CENESP,FELIPE BARBOSA GONÇALVES,TI | SISTEMAS,Não atribuído,2026-05-18 10:36:54,2026-05-18 10:48:55,2026-05-18 10:51:54,2026-05-18 13:36:54,2026-05-18 10:48:55,NaT,No prazo,2026-05-18 00:15:00,inbrands.com.br,NaN,TI,SISTEMAS,Lentidão Linx,Fechado,-10.365000,0.200278,FINALIZADO,NaN,0.0,ATÉ 1H,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27072,71,salinas.higienopolis@inbrands.com.br,Notas Fiscais Não Aparecem Para Entrada.,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 18:21:29,2025-04-10 12:04:27,2025-04-08 20:21:29,2025-04-09 21:21:29,2025-04-08 18:59:35,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,NaN,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9703.641944,0.635000,FINALIZADO,24.0,1.0,ATÉ 1H,0
27073,70,salinas.higienopolis@inbrands.com.br,Nota Fiscal De Doação,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 17:55:21,2025-04-10 12:04:27,2025-04-08 19:55:21,2025-04-09 20:55:21,2025-04-08 18:58:27,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,NaN,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9704.077500,1.051667,FINALIZADO,24.0,1.0,1H A 4H,0
27074,69,salinas.iguatemi@inbrands.com.br,Peça Não Cadastrada,FECHADO,Requisição,Formcreator,MUITO BAIXA,Não Atribuído,Não Atribuído,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 15

In [143]:
# Tratamento do campo nome tecnico e grupo

df['nome_tecnico'] = df['nome_tecnico'].astype(str)
df['grupo'] = df['grupo'].astype(str)


df[['departamento', 'grupo_descricao']] = (
    df['grupo']
    .str.split('|', n=1, expand=True)
)

df['nome_tecnico'] = df['nome_tecnico'].str.strip()
df['departamento'] = df['departamento'].str.strip()
df['grupo_descricao'] = df['grupo_descricao'].str.strip()

df

,id,email_requerente,titulo,status,tipo,origem_abertura,prioridade,categoria,categoria_completa,localizacao,nome_tecnico,grupo,tempo_chamado,data_criacao,ultima_atualizacao,tempo_atribuicao,tempo_solucao,data_solucao,data_fechamento,expirado,tempo_resposta,dominio_email,codigo_unidade,departamento,grupo_descricao,subcategoria,status_macro,tempo_primeira_resposta_horas,tempo_resolucao_horas,status_sla,sla_previsto_horas,flag_sla_dentro_prazo,faixa_sla,flag_sem_tecnico
0,27636,aira.coelho@inbrands.com.br,Reparos De Pintura Escritório,FECHADO,Requisição,Formcreator,MÉDIA,Pintura - Hub,Pintura - Hub > Reparos Escritório,HUB-SP - CENESP,NaN,MANUTENÇÃO | HUB,Não atribuído,2026-05-18 10:59:13,2026-05-18 10:59:28,NaT,2026-05-25 10:59:13,2026-05-18 10:59:25,2026-05-18 10:59:28,No prazo,NaT,inbrands.com.br,NaN,MANUTENÇÃO,HUB,Reparos Escritório,Fechado,NaN,0.003333,FINALIZADO,8.0,1.0,ATÉ 1H,0
1,27635,keylla.jesus@inbrands.com.br,Cadastro Liberado No Linx E No Wms Segue Bloqueado,NOVO,Incidente,Formcreator,ALTA,Linx,Linx > Erro Em Tela No Linx,CD EMBU,NaN,TI | SISTEMAS,Não atribuído,2026-05-18 10:48:21,2026-05-18 10:48:21,2026-05-18 11:18:21,2026-05-18 17:48:21,NaT,NaT,No prazo,2026-05-18 00:30:00,inbrands.com.br,NaN,TI,SISTEMAS,Erro Em Tela No Linx,Aberto,-10.305833,NaN,NÃO CLASSIFICADO,4.0,NaN,EM ANDAMENTO,0
2,27634,tommy.fortaleza@inbrands.com.br,A Pagina Do General/ Lançamentos Não Esta Abrindo:,NOVO,Incidente,Formcreator,BAIXA,Relatórios,Relatórios > Relatório Não Enviado,TH - OUTLET FORTALEZA,NaN,TI | SISTEMAS,Não atribuído,2026-05-18 10:47:46,2026-05-18 10:47:46,2026-05-18 13:47:46,2026-05-20 08:47:46,NaT,NaT,No prazo,2026-05-18 03:00:00,inbrands.com.br,NaN,TI,SISTEMAS,Relatório Não Enviado,Aberto,-7.796111,NaN,NÃO CLASSIFICADO,24.0,NaN,EM ANDAMENTO,0
3,27633,vr.portoalegre@inbrands.com.br,Lâmpada Queimada,SOLUCIONADO,Incidente,Formcreator,ALTA,Iluminação - Lojas,Iluminação - Lojas > Totalmente Apagada,VR - PORTO ALEGRE,NaN,MANUTENÇÃO | LOJAS,Não atribuído,2026-05-18 10:45:45,2026-05-18 10:51:00,NaT,2026-05-20 17:45:45,2026-05-18 10:51:00,NaT,No prazo,NaT,inbrands.com.br,NaN,MANUTENÇÃO,LOJAS,Totalmente Apagada,Fechado,NaN,0.087500,FINALIZADO,4.0,1.0,ATÉ 1H,0
4,27632,joao.lacerda@inbrands.com.br,Linx Travado,SOLUCIONADO,Incidente,Formcreator,MUITO ALTA,Linx,Linx > Lentidão Linx,HUB-SP - CENESP,FELIPE BARBOSA GONÇALVES,TI | SISTEMAS,Não atribuído,2026-05-18 10:36:54,2026-05-18 10:48:55,2026-05-18 10:51:54,2026-05-18 13:36:54,2026-05-18 10:48:55,NaT,No prazo,2026-05-18 00:15:00,inbrands.com.br,NaN,TI,SISTEMAS,Lentidão Linx,Fechado,-10.365000,0.200278,FINALIZADO,NaN,0.0,ATÉ 1H,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27072,71,salinas.higienopolis@inbrands.com.br,Notas Fiscais Não Aparecem Para Entrada.,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 18:21:29,2025-04-10 12:04:27,2025-04-08 20:21:29,2025-04-09 21:21:29,2025-04-08 18:59:35,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,NaN,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9703.641944,0.635000,FINALIZADO,24.0,1.0,ATÉ 1H,0
27073,70,salinas.higienopolis@inbrands.com.br,Nota Fiscal De Doação,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 17:55:21,2025-04-10 12:04:27,2025-04-08 19:55:21,2025-04-09 20:55:21,2025-04-08 18:58:27,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,NaN,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9704.077500,1.051667,FINALIZADO,24.0,1.0,1H A 4H,0
27074,69,salinas.iguatemi@inbrands.com.br,Peça Não Cadastrada,FECHADO,Requisição,Formcreator,MUITO BAIXA,Não Atribuído,Não Atribuído,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 15

In [144]:
df = df.replace(pd.NaT, "")

# Tratamento do campo Categoria
df['categoria'] = df['categoria'].astype(str)

df[['categoria', 'subcategoria']] = (
    df['categoria_completa']
    .str.split('>', n=1, expand=True)
)

# Remover espaços extras e dados nulos
df['categoria'] = df['categoria'].str.strip()
df['subcategoria'] = df['subcategoria'].str.strip().fillna('Não atribuído')

df

,id,email_requerente,titulo,status,tipo,origem_abertura,prioridade,categoria,categoria_completa,localizacao,nome_tecnico,grupo,tempo_chamado,data_criacao,ultima_atualizacao,tempo_atribuicao,tempo_solucao,data_solucao,data_fechamento,expirado,tempo_resposta,dominio_email,codigo_unidade,departamento,grupo_descricao,subcategoria,status_macro,tempo_primeira_resposta_horas,tempo_resolucao_horas,status_sla,sla_previsto_horas,flag_sla_dentro_prazo,faixa_sla,flag_sem_tecnico
0,27636,aira.coelho@inbrands.com.br,Reparos De Pintura Escritório,FECHADO,Requisição,Formcreator,MÉDIA,Pintura - Hub,Pintura - Hub > Reparos Escritório,HUB-SP - CENESP,,MANUTENÇÃO | HUB,Não atribuído,2026-05-18 10:59:13,2026-05-18 10:59:28,NaT,2026-05-25 10:59:13,2026-05-18 10:59:25,2026-05-18 10:59:28,No prazo,NaT,inbrands.com.br,,MANUTENÇÃO,HUB,Reparos Escritório,Fechado,NaN,0.003333,FINALIZADO,8.0,1.0,ATÉ 1H,0
1,27635,keylla.jesus@inbrands.com.br,Cadastro Liberado No Linx E No Wms Segue Bloqueado,NOVO,Incidente,Formcreator,ALTA,Linx,Linx > Erro Em Tela No Linx,CD EMBU,,TI | SISTEMAS,Não atribuído,2026-05-18 10:48:21,2026-05-18 10:48:21,2026-05-18 11:18:21,2026-05-18 17:48:21,NaT,NaT,No prazo,2026-05-18 00:30:00,inbrands.com.br,,TI,SISTEMAS,Erro Em Tela No Linx,Aberto,-10.305833,NaN,NÃO CLASSIFICADO,4.0,NaN,EM ANDAMENTO,0
2,27634,tommy.fortaleza@inbrands.com.br,A Pagina Do General/ Lançamentos Não Esta Abrindo:,NOVO,Incidente,Formcreator,BAIXA,Relatórios,Relatórios > Relatório Não Enviado,TH - OUTLET FORTALEZA,,TI | SISTEMAS,Não atribuído,2026-05-18 10:47:46,2026-05-18 10:47:46,2026-05-18 13:47:46,2026-05-20 08:47:46,NaT,NaT,No prazo,2026-05-18 03:00:00,inbrands.com.br,,TI,SISTEMAS,Relatório Não Enviado,Aberto,-7.796111,NaN,NÃO CLASSIFICADO,24.0,NaN,EM ANDAMENTO,0
3,27633,vr.portoalegre@inbrands.com.br,Lâmpada Queimada,SOLUCIONADO,Incidente,Formcreator,ALTA,Iluminação - Lojas,Iluminação - Lojas > Totalmente Apagada,VR - PORTO ALEGRE,,MANUTENÇÃO | LOJAS,Não atribuído,2026-05-18 10:45:45,2026-05-18 10:51:00,NaT,2026-05-20 17:45:45,2026-05-18 10:51:00,NaT,No prazo,NaT,inbrands.com.br,,MANUTENÇÃO,LOJAS,Totalmente Apagada,Fechado,NaN,0.087500,FINALIZADO,4.0,1.0,ATÉ 1H,0
4,27632,joao.lacerda@inbrands.com.br,Linx Travado,SOLUCIONADO,Incidente,Formcreator,MUITO ALTA,Linx,Linx > Lentidão Linx,HUB-SP - CENESP,FELIPE BARBOSA GONÇALVES,TI | SISTEMAS,Não atribuído,2026-05-18 10:36:54,2026-05-18 10:48:55,2026-05-18 10:51:54,2026-05-18 13:36:54,2026-05-18 10:48:55,NaT,No prazo,2026-05-18 00:15:00,inbrands.com.br,,TI,SISTEMAS,Lentidão Linx,Fechado,-10.365000,0.200278,FINALIZADO,NaN,0.0,ATÉ 1H,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27072,71,salinas.higienopolis@inbrands.com.br,Notas Fiscais Não Aparecem Para Entrada.,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 18:21:29,2025-04-10 12:04:27,2025-04-08 20:21:29,2025-04-09 21:21:29,2025-04-08 18:59:35,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9703.641944,0.635000,FINALIZADO,24.0,1.0,ATÉ 1H,0
27073,70,salinas.higienopolis@inbrands.com.br,Nota Fiscal De Doação,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 17:55:21,2025-04-10 12:04:27,2025-04-08 19:55:21,2025-04-09 20:55:21,2025-04-08 18:58:27,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9704.077500,1.051667,FINALIZADO,24.0,1.0,1H A 4H,0
27074,69,salinas.iguatemi@inbrands.com.br,Peça Não Cadastrada,FECHADO,Requisição,Formcreator,MUITO BAIXA,Não Atribuído,Não Atribuído,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 15:37:26,2025-07-16 16:10:09,2025-0

In [145]:
# Criando a coluna de Status_macro

df['status'] = (df['status'].astype(str)
                .str.upper()
                .str.strip()
                )

map_status_macro = {'NOVO': 'Aberto',
                    'EM ATENDIMENTO (ATRIBUÍDO)': 'Em atendimento', 
                    'EM ATENDIMENTO (PLANEJADO)': 'Em atendimento',
                    'FECHADO': 'Fechado',
                    'SOLUCIONADO': 'Fechado',
                    'PENDENTE': 'Pendente',
                    }

df['status_macro'] = (df['status'].map(map_status_macro))

df

,id,email_requerente,titulo,status,tipo,origem_abertura,prioridade,categoria,categoria_completa,localizacao,nome_tecnico,grupo,tempo_chamado,data_criacao,ultima_atualizacao,tempo_atribuicao,tempo_solucao,data_solucao,data_fechamento,expirado,tempo_resposta,dominio_email,codigo_unidade,departamento,grupo_descricao,subcategoria,status_macro,tempo_primeira_resposta_horas,tempo_resolucao_horas,status_sla,sla_previsto_horas,flag_sla_dentro_prazo,faixa_sla,flag_sem_tecnico
0,27636,aira.coelho@inbrands.com.br,Reparos De Pintura Escritório,FECHADO,Requisição,Formcreator,MÉDIA,Pintura - Hub,Pintura - Hub > Reparos Escritório,HUB-SP - CENESP,,MANUTENÇÃO | HUB,Não atribuído,2026-05-18 10:59:13,2026-05-18 10:59:28,NaT,2026-05-25 10:59:13,2026-05-18 10:59:25,2026-05-18 10:59:28,No prazo,NaT,inbrands.com.br,,MANUTENÇÃO,HUB,Reparos Escritório,Fechado,NaN,0.003333,FINALIZADO,8.0,1.0,ATÉ 1H,0
1,27635,keylla.jesus@inbrands.com.br,Cadastro Liberado No Linx E No Wms Segue Bloqueado,NOVO,Incidente,Formcreator,ALTA,Linx,Linx > Erro Em Tela No Linx,CD EMBU,,TI | SISTEMAS,Não atribuído,2026-05-18 10:48:21,2026-05-18 10:48:21,2026-05-18 11:18:21,2026-05-18 17:48:21,NaT,NaT,No prazo,2026-05-18 00:30:00,inbrands.com.br,,TI,SISTEMAS,Erro Em Tela No Linx,Aberto,-10.305833,NaN,NÃO CLASSIFICADO,4.0,NaN,EM ANDAMENTO,0
2,27634,tommy.fortaleza@inbrands.com.br,A Pagina Do General/ Lançamentos Não Esta Abrindo:,NOVO,Incidente,Formcreator,BAIXA,Relatórios,Relatórios > Relatório Não Enviado,TH - OUTLET FORTALEZA,,TI | SISTEMAS,Não atribuído,2026-05-18 10:47:46,2026-05-18 10:47:46,2026-05-18 13:47:46,2026-05-20 08:47:46,NaT,NaT,No prazo,2026-05-18 03:00:00,inbrands.com.br,,TI,SISTEMAS,Relatório Não Enviado,Aberto,-7.796111,NaN,NÃO CLASSIFICADO,24.0,NaN,EM ANDAMENTO,0
3,27633,vr.portoalegre@inbrands.com.br,Lâmpada Queimada,SOLUCIONADO,Incidente,Formcreator,ALTA,Iluminação - Lojas,Iluminação - Lojas > Totalmente Apagada,VR - PORTO ALEGRE,,MANUTENÇÃO | LOJAS,Não atribuído,2026-05-18 10:45:45,2026-05-18 10:51:00,NaT,2026-05-20 17:45:45,2026-05-18 10:51:00,NaT,No prazo,NaT,inbrands.com.br,,MANUTENÇÃO,LOJAS,Totalmente Apagada,Fechado,NaN,0.087500,FINALIZADO,4.0,1.0,ATÉ 1H,0
4,27632,joao.lacerda@inbrands.com.br,Linx Travado,SOLUCIONADO,Incidente,Formcreator,MUITO ALTA,Linx,Linx > Lentidão Linx,HUB-SP - CENESP,FELIPE BARBOSA GONÇALVES,TI | SISTEMAS,Não atribuído,2026-05-18 10:36:54,2026-05-18 10:48:55,2026-05-18 10:51:54,2026-05-18 13:36:54,2026-05-18 10:48:55,NaT,No prazo,2026-05-18 00:15:00,inbrands.com.br,,TI,SISTEMAS,Lentidão Linx,Fechado,-10.365000,0.200278,FINALIZADO,NaN,0.0,ATÉ 1H,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27072,71,salinas.higienopolis@inbrands.com.br,Notas Fiscais Não Aparecem Para Entrada.,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 18:21:29,2025-04-10 12:04:27,2025-04-08 20:21:29,2025-04-09 21:21:29,2025-04-08 18:59:35,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9703.641944,0.635000,FINALIZADO,24.0,1.0,ATÉ 1H,0
27073,70,salinas.higienopolis@inbrands.com.br,Nota Fiscal De Doação,FECHADO,Requisição,Formcreator,BAIXA,Fiscal,Fiscal > Reprocessamento De Notas Fiscais,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 17:55:21,2025-04-10 12:04:27,2025-04-08 19:55:21,2025-04-09 20:55:21,2025-04-08 18:58:27,2025-04-10 12:04:27,Vencido,2026-05-18 02:00:00,inbrands.com.br,,TI,SUPORTE A LOJAS,Reprocessamento De Notas Fiscais,Fechado,9704.077500,1.051667,FINALIZADO,24.0,1.0,1H A 4H,0
27074,69,salinas.iguatemi@inbrands.com.br,Peça Não Cadastrada,FECHADO,Requisição,Formcreator,MUITO BAIXA,Não Atribuído,Não Atribuído,LOJAS,THAINA CARVALHO DO NASCIMENTO,TI | SUPORTE A LOJAS,Não atribuído,2025-04-08 15:37:26,2025-07-16 16:10:09,2025-0

## Métricas

In [146]:
### Tempo Primeira Resposta
df['tempo_primeira_resposta_horas'] = (
    (
        df['tempo_resposta']
        - df['data_criacao']
    )
    .dt.total_seconds()
    / 3600
)

In [147]:
### Tempo Resolução
df['tempo_resolucao_horas'] = (
    (
        df['data_solucao']
        - df['data_criacao']
    )
    .dt.total_seconds()
    / 3600
)

## SLA (Regra Corporativa)

In [148]:
condicoes = [

    df['data_solucao'].notnull(),

    (
        df['data_solucao'].isnull()
        &
        df['status'].isin([
            'ABERTO',
            'EM ANDAMENTO'
        ])
    ),

    (
        df['data_solucao'].isnull()
        &
        df['status'].str.contains(
            'PENDENTE',
            na=False
        )
    ),

    (
        df['nome_tecnico'].isnull()
    )
]

In [149]:
valores = [
    'FINALIZADO',
    'EM ABERTO',
    'PENDENTE',
    'NÃO ATRIBUÍDO'
]

In [150]:
df['status_sla'] = np.select(
    condicoes,
    valores,
    default='NÃO CLASSIFICADO'
)

In [151]:
map_sla = {
    'CRÍTICA': 2,
    'ALTA': 4,
    'MÉDIA': 8,
    'BAIXA': 24
}

In [152]:
df['sla_previsto_horas'] = (
    df['prioridade']
    .map(map_sla)
)

In [153]:
df['flag_sla_dentro_prazo'] = np.where(

    df['tempo_resolucao_horas'].isnull(),

    np.nan,

    np.where(
        df['tempo_resolucao_horas']
        <=
        df['sla_previsto_horas'],
        1,
        0
    )
)

In [154]:
def classificar_faixa_sla(tempo):

    if pd.isnull(tempo):
        return 'EM ANDAMENTO'

    elif tempo <= 1:
        return 'ATÉ 1H'

    elif tempo <= 4:
        return '1H A 4H'

    elif tempo <= 8:
        return '4H A 8H'

    elif tempo <= 18:
        return '8H A 18H'

    else:
        return 'ACIMA 24H'

In [155]:
df['faixa_sla'] = (
    df['tempo_resolucao_horas']
    .apply(classificar_faixa_sla)
)

## Criação das Tabelas de Dimensão

In [174]:
### 5.1 D_DATA
D_DATA = pd.DataFrame({
    'data_completa': calendario
})

D_DATA['data_merge'] = (
    D_DATA['data_completa']
    .dt.date
)

D_DATA['data_sk'] = (
    D_DATA['data_completa']
    .dt.strftime('%Y%m%d')
    .astype(int)
)

### Colunas Temporais ###

D_DATA['ano'] = (
    D_DATA['data_completa']
    .dt.year
)

D_DATA['mes'] = (
    D_DATA['data_completa']
    .dt.month
)

D_DATA['nome_mes'] = (
    D_DATA['data_completa']
    .dt.month_name()
)

D_DATA['trimestre'] = (
    D_DATA['data_completa']
    .dt.quarter
)

D_DATA['dia'] = (
    D_DATA['data_completa']
    .dt.day
)

D_DATA['dia_semana_num'] = (
    D_DATA['data_completa']
    .dt.dayofweek
)

D_DATA['dia_semana'] = (
    D_DATA['data_completa']
    .dt.day_name()
)

D_DATA['dia_semana'] = (
    D_DATA['dia_semana']
    .map(map_dias)
)

D_DATA['eh_dia_util'] = np.where(
    D_DATA['dia_semana_num'] < 5,
    1,
    0
)


In [157]:
### 5.2 D_REQUERENTE
D_REQUERENTE = (
        df[['email_requerente', 'dominio_email']]
        .drop_duplicates()
        .sort_values(by='email_requerente')
        .reset_index(drop=True)
)
D_REQUERENTE['requerente_sk'] = D_REQUERENTE.index + 1
D_REQUERENTE

,email_requerente,dominio_email,requerente_sk
0,,Não atribuído,1
1,Breno Maciel,Não atribuído,2
2,Felipe Gonçalves,Não atribuído,3
3,Gabriele Bispo,Não atribuído,4
4,Jefferson Barbosa,Não atribuído,5
...,...,...,...
969,yellen.moreira@inbrands.com.br,inbrands.com.br,970
970,ygor.parecy@inbrands.com.br,inbrands.com.br,971
971,yngrid.ferreira@inbrands.com.br,inbrands.com.br,972
972,yuki.sato@inbrands.com.br,inbrands.com.br,973


In [158]:
### 5.3 D_TECNICO
D_TECNICO = (
    df[
        df['nome_tecnico'].notnull()
    ][['nome_tecnico', 'departamento']]
    .drop_duplicates()
    .sort_values(by='nome_tecnico')
    .reset_index(drop=True)
)

# Criação da flag_sem_tecnico
df['flag_sem_tecnico'] = np.where(
    df['nome_tecnico'].isnull(),
    1,
    0
)

# Validar quantidade sem técnico
if df['flag_sem_tecnico'].sum() > 0:
    print("Sem Técnico")

df_sem_tecnico = df[
    df['flag_sem_tecnico'] == 1
]
D_TECNICO['tecnico_sk'] = D_TECNICO.index + 1
D_TECNICO

,nome_tecnico,departamento,tecnico_sk
0,,MANUTENÇÃO,1
1,,GPP / SEGURANÇA (HUB),2
2,,,3
3,,GPP / SEGURANÇA (LOJAS),4
4,,ADMINISTRAÇÃO DE PESSOAL,5
...,...,...,...
63,VALDIR BOSCO DA SILVA JUNIOR,TI,64
64,VANUSA PEREIRA SALES,ADMINISTRAÇÃO DE PESSOAL,65
65,WESLLEY DE SANTANA MOREIRA,CROWN IT N2,66
66,WESLLEY DE SANTANA MOREIRA,TI,67


In [159]:
### 5.5 D_CATEGORIA
D_CATEGORIA = (
    df[['categoria', 'subcategoria']]
    .drop_duplicates()
    .sort_values(by='categoria')
    .reset_index(drop=True)
)
D_CATEGORIA['categoria_sk'] = D_CATEGORIA.index + 1
D_CATEGORIA

,categoria,subcategoria,categoria_sk
0,Abertura Via Chat Indevida,Não atribuído,1
1,Acessos,Não atribuído,2
2,Acessos,Acesso Ao Ts,3
3,Acessos,Wi-Fi,4
4,Acessos,Banco De Dados,5
...,...,...,...
514,Verdanadesk,Inclusão De Usuário,515
515,Verdanadesk,Ajustes,516
516,Verdanadesk,Perfil,517
517,Verdanadesk,Criação De Categoria,518


In [160]:
### 5.6 D_GRUPO

D_GRUPO = (
    df[['grupo']]
    .drop_duplicates()
)

D_GRUPO['grupo'] = (
    D_GRUPO['grupo']
    .astype(str)
    .str.upper()
    .str.strip()
)

D_GRUPO['area_responsavel'] = (
    D_GRUPO['grupo']
    .map(map_area)
)

D_GRUPO['torre_servico'] = (
    D_GRUPO['grupo']
    .map(map_torre)
)

D_GRUPO = (
    D_GRUPO
    .sort_values(by='grupo')
    .reset_index(drop=True)
)

D_GRUPO['grupo_sk'] = (
    D_GRUPO.index + 1
)


In [161]:
### 5.7 D_STATUS
D_STATUS = (
    df[['status', 'status_macro']]
    .drop_duplicates()
    .sort_values(by='status')
    .reset_index(drop=True)
)
D_STATUS['status_sk'] = D_STATUS.index + 1
D_STATUS

,status,status_macro,status_sk
0,EM ATENDIMENTO (ATRIBUÍDO),Em atendimento,1
1,EM ATENDIMENTO (PLANEJADO),Em atendimento,2
2,FECHADO,Fechado,3
3,NOVO,Aberto,4
4,PENDENTE,Pendente,5
5,SOLUCIONADO,Fechado,6


In [162]:
### 5.8 D_TIPO
D_TIPO = (
    df[['tipo']]
    .drop_duplicates()
    .sort_values(by='tipo')
    .reset_index(drop=True)
)
D_TIPO['tipo_sk'] = D_TIPO.index + 1
D_TIPO

,tipo,tipo_sk
0,Incidente,1
1,Requisição,2


In [163]:
### 5.9 D_ORIGEM_ABERTURA
D_ORIGEM_ABERTURA = (
    df[['origem_abertura']]
    .drop_duplicates()
    .sort_values(by='origem_abertura')
    .reset_index(drop=True)
)
D_ORIGEM_ABERTURA['origem_abertura_sk'] = D_ORIGEM_ABERTURA.index + 1
D_ORIGEM_ABERTURA

,origem_abertura,origem_abertura_sk
0,Chat,1
1,Direct,2
2,E-Mail,3
3,Formcreator,4
4,Helpdesk,5
5,Other,6


In [164]:
### 5.10 D_PRIORIDADE
D_PRIORIDADE = (
    df[['prioridade']]
    .drop_duplicates()
    .sort_values(by='prioridade')
    .reset_index(drop=True)
)
D_PRIORIDADE['prioridade_sk'] = D_PRIORIDADE.index + 1
D_PRIORIDADE

,prioridade,prioridade_sk
0,ALTA,1
1,BAIXA,2
2,CRÍTICA,3
3,MUITO ALTA,4
4,MUITO BAIXA,5
5,MÉDIA,6


In [165]:
### 5.11 D_SLA
D_SLA = (
    df[
        [
            'sla_previsto_horas',
            'status_sla',
            'faixa_sla'
        ]
    ]
    .drop_duplicates()
    .sort_values(
        by=['sla_previsto_horas']
    )
    .reset_index(drop=True)
)
D_SLA['sla_sk'] = (
    D_SLA.index + 1
)

## Criação da Tabela Fato 

In [176]:
F_CHAMADO = df.merge(

    D_DATA[
        [
            'data_sk',
            'data_merge'
        ]
    ],

    left_on='data_abertura_merge',
    right_on='data_merge',

    how='left'
)

F_CHAMADO = F_CHAMADO.rename(
    columns={
        'data_sk': 'data_abertura_sk'
    }
)

In [177]:
F_CHAMADO = F_CHAMADO.merge(

    D_DATA[
        [
            'data_sk',
            'data_merge'
        ]
    ],

    left_on='data_fechamento_merge',
    right_on='data_merge',

    how='left'
)

In [178]:
F_CHAMADO = F_CHAMADO.rename(
    columns={
        'data_sk': 'data_fechamento_sk'
    }
)

In [179]:
# Criação da Tabela fato

F_CHAMADOS = df.merge(
    D_TECNICO,
    how='left',
    left_on=['nome_tecnico', 'departamento'],
    right_on=['nome_tecnico', 'departamento']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_CATEGORIA,
    how='left',
    left_on=['categoria', 'subcategoria'],
    right_on=['categoria', 'subcategoria']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_STATUS,
    how='left',
    left_on=['status','status_macro'],
    right_on=['status','status_macro']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_REQUERENTE,
    how='left',
    left_on=['email_requerente', 'dominio_email'],
    right_on=['email_requerente', 'dominio_email']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_TIPO,
    how='left',
    left_on=['tipo'],
    right_on=['tipo']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_ORIGEM_ABERTURA,
    how='left',
    left_on=['origem_abertura'],
    right_on=['origem_abertura']
)

F_CHAMADOS = F_CHAMADOS.merge(
    D_GRUPO,
    how='left',
    left_on=['grupo'],
    right_on=['grupo']
)

F_CHAMADOS = F_CHAMADOS.merge(

    D_SLA[
        [
            'sla_sk',
            'sla_previsto_horas',
            'status_sla',
            'faixa_sla'
        ]
    ],

    on=[
        'sla_previsto_horas',
        'status_sla',
        'faixa_sla'
    ],

    how='left'
)

F_CHAMADOS = F_CHAMADOS.merge(

    D_DATA[
        [
            'data_sk',
            'data_merge'
        ]
    ],

    left_on='data_fechamento_merge',
    right_on='data_merge',

    how='left'
)

F_CHAMADOS = F_CHAMADOS.drop(columns=
                             ['nome_tecnico', 
                              'grupo',
                              'departamento', 
                              'categoria', 
                              'categoria', 
                              'subcategoria',   
                              'status',
                              'email_requerente',
                              'tipo',
                              'origem_abertura',
                              'prioridade',
                              'localizacao',
                              'categoria_completa',
                              'tempo_chamado',
                              'data_criacao',
                              'ultima_atualizacao',
                              'tempo_atribuicao',
                              'tempo_solucao',
                              'data_fechamento',
                              'data_solucao',
                              'expirado',
                              'tempo_resposta',
                              'grupo_descricao',
                              'dominio_email',
                              'status_macro']
                              )
F_CHAMADOS = F_CHAMADOS.rename(columns={'id': 'id_chamado'})
F_CHAMADOS['chamado_sk'] = F_CHAMADOS.index + 1
F_CHAMADOS

,id_chamado,titulo,codigo_unidade,tempo_primeira_resposta_horas,tempo_resolucao_horas,status_sla,sla_previsto_horas,flag_sla_dentro_prazo,faixa_sla,flag_sem_tecnico,data_abertura_merge,data_fechamento_merge,tecnico_sk,categoria_sk,status_sk,requerente_sk,tipo_sk,origem_abertura_sk,area_responsavel,torre_servico,grupo_sk,sla_sk,data_sk,data_merge,chamado_sk
0,27636,Reparos De Pintura Escritório,,NaN,0.003333,FINALIZADO,8.0,1.0,ATÉ 1H,0,2026-05-18,2026-05-18,1,393,3,12,2,4,NaN,NaN,8,12,NaN,NaN,1
1,27635,Cadastro Liberado No Linx E No Wms Segue Bloqueado,,-10.305833,NaN,NÃO CLASSIFICADO,4.0,NaN,EM ANDAMENTO,0,2026-05-18,NaT,7,310,4,565,1,4,NaN,SISTEMAS,18,2,NaN,NaN,2
2,27634,A Pagina Do General/ Lançamentos Não Esta Abrindo:,,-7.796111,NaN,NÃO CLASSIFICADO,24.0,NaN,EM ANDAMENTO,0,2026-05-18,NaT,7,448,4,873,1,4,NaN,SISTEMAS,18,16,NaN,NaN,3
3,27633,Lâmpada Queimada,,NaN,0.087500,FINALIZADO,4.0,1.0,ATÉ 1H,0,2026-05-18,NaT,1,270,6,948,1,4,NaN,NaN,9,4,NaN,NaN,4
4,27632,Linx Travado,,-10.365000,0.200278,FINALIZADO,NaN,0.0,ATÉ 1H,0,2026-05-18,NaT,27,307,6,521,1,4,NaN,SISTEMAS,18,23,NaN,NaN,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27072,71,Notas Fiscais Não Aparecem Para Entrada.,,9703.641944,0.635000,FINALIZADO,24.0,1.0,ATÉ 1H,0,2025-04-08,2025-04-10,61,217,3,820,2,4,TI,SUPORTE LOJAS,20,20,20250410.0,2025-04-10,27073
27073,70,Nota Fiscal De Doação,,9704.077500,1.051667,FINALIZADO,24.0,1.0,1H A 4H,0,2025-04-08,2025-04-10,61,217,3,820,2,4,TI,SUPORTE LOJAS,20,19,20250410.0,2025-04-10,27074
27074,69,Peça Não Cadastrada,,9707.376111,51.071667,FINALIZADO,NaN,0.0,ACIMA 24H,0,2025-04-08,2025-04-12,61,363,3,821,2,4,TI,SUPORTE LOJAS,20,25,20250412.0,2025-04-12,27075
27075,68,Estou Precisando Passar Venda No Linxpos E A Tela De Vendas Não Está Abrindo. Parou De Funcionar Hoje A Tarde Depois De Uma Atualização Feira Por Aí Mesmo.,,NaN,14.479722,FINALIZADO,8.0,0.0,8H A 18H,0,2025-04-07,2025-04-10,33,327,3,392,1,1,TI,SUPORTE LOJAS,20,14,20250410.0,2025-04-10,27076


## Exportação via Excel

In [180]:
# Criar arquivos excel

with pd.ExcelWriter("Tratamento_chamados.xlsx") as writer:
    D_TECNICO.to_excel(writer, sheet_name="D_TECNICO", index=False)
    D_CATEGORIA.to_excel(writer, sheet_name="D_CATEGORIA", index=False)
    D_STATUS.to_excel(writer, sheet_name="D_STATUS", index=False)
    D_REQUERENTE.to_excel(writer, sheet_name="D_REQUERENTE", index=False)
    F_CHAMADOS.to_excel(writer, sheet_name="F_CHAMADOS", index=False)
    D_TIPO.to_excel(writer, sheet_name="D_TIPO", index=False)
    D_ORIGEM_ABERTURA.to_excel(writer, sheet_name="D_ORIGEM_ABERTURA", index=False)
    D_PRIORIDADE.to_excel(writer, sheet_name="D_PRIORIDADE", index=False)
    D_GRUPO.to_excel(writer, sheet_name="D_GRUPO", index=False)
    D_SLA.to_excel(writer, sheet_name="D_SLA", index=False)
    D_DATA.to_excel(writer, sheet_name="D_DATA", index=False)